In [ ]:
# ==========================================
# 6. EXPLORE PLANET 8-BAND DATA
# Mirrors the Maxar RGB exploration in 9-explore_max_min.ipynb:
# sample pixels, compute mean/std/mean+3std, plot histograms per band.
# ==========================================

import glob
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import os

# EDIT: point this at your own directory of 8-band Planet SuperDove SR scenes
INPUT_DIR = "./data/planet_scenes/"

# Standard PSB.SD (SuperDove) 8-band order, confirmed against this dataset's
# XML metadata (bandNumber 1-8 in this sequence)
PLANET_BAND_NAMES = [
    'Coastal Blue', 'Blue', 'Green I', 'Green', 'Yellow', 'Red', 'Red Edge', 'NIR'
]

def sample_planet_band_stats(tif_paths, n_bands=8, sample_size=1_000_000, nodata=0, seed=0):
    """
    Randomly sample up to `sample_size` valid (non-nodata) pixels per band,
    pooled evenly across all files in tif_paths.
    """
    rng = np.random.default_rng(seed)
    per_file_budget = max(1, sample_size // len(tif_paths))
    band_samples = [[] for _ in range(n_bands)]

    for tif_path in tif_paths:
        with rasterio.open(tif_path) as src:
            arr = src.read(list(range(1, n_bands + 1))).astype(np.float32)  # (n_bands, H, W)

        for b in range(n_bands):
            band = arr[b].ravel()
            valid = band[band != nodata]
            if valid.size == 0:
                continue
            if valid.size > per_file_budget:
                idx = rng.choice(valid.size, per_file_budget, replace=False)
                valid = valid[idx]
            band_samples[b].append(valid)

        print(f"  sampled {os.path.basename(tif_path)} ({arr.shape[1]}x{arr.shape[2]})")

    return [np.concatenate(b) if b else np.array([]) for b in band_samples]


def summarize_and_plot_band_stats(band_samples, band_names, bins=60, figsize=(24, 10)):
    stats = []
    n_bands = len(band_samples)
    ncols = 4
    nrows = int(np.ceil(n_bands / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)

    for b in range(n_bands):
        data = band_samples[b]
        ax = axes[b]
        if data.size == 0:
            ax.set_title(f'Band {b + 1}: {band_names[b]}\nNo data')
            ax.axis('off')
            continue

        mean_val = float(np.mean(data))
        std_val = float(np.std(data))
        mean_plus_3std = mean_val + 3 * std_val
        min_val = float(np.min(data))
        max_val = float(np.max(data))

        stats.append({
            'Band': b + 1,
            'Name': band_names[b],
            'Min': min_val,
            'Max': max_val,
            'Mean': mean_val,
            'Std Dev': std_val,
            'Mean+3Std': mean_plus_3std,
            'N Sampled': data.size,
        })

        ax.hist(data, bins=bins, alpha=0.7, edgecolor='black', linewidth=0.3)
        ax.axvline(mean_val, color='darkred', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
        ax.axvline(mean_plus_3std, color='orange', linestyle='-.', linewidth=2, label=f'Mean+3σ: {mean_plus_3std:.1f}')
        ax.set_title(f'Band {b + 1}: {band_names[b]}', fontweight='bold')
        ax.set_xlabel('DN value')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    for b in range(n_bands, len(axes)):
        axes[b].axis('off')

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(stats)


# Grab all 8-band surface reflectance Planet scenes in the input directory
planet_8b_files = sorted(glob.glob(os.path.join(INPUT_DIR, '*_8b_sr.tif')))
print(f"Found {len(planet_8b_files)} 8-band Planet SR files")

planet_band_samples = sample_planet_band_stats(planet_8b_files, n_bands=8, sample_size=1_000_000)
planet_band_stats_df = summarize_and_plot_band_stats(planet_band_samples, PLANET_BAND_NAMES)
print(planet_band_stats_df.to_string(index=False))

In [ ]:
# ==========================================
# 7. SAVE PLANET 8-BAND STATS FOR LATER REFERENCE
# Follows the same per-array convention as numpy_stats/ in boreal_chm_dino
# (channel_means_{desc}.npy, channel_stds_{desc}.npy, ...) plus a combined
# dict .npy for convenience.
# ==========================================

STATS_DIR = "./numpy_stats/"
os.makedirs(STATS_DIR, exist_ok=True)

stats_desc = "planet_8b_sr"  # change this if you re-run with a different sample_size/subset

channel_means = planet_band_stats_df['Mean'].to_numpy()
channel_stds = planet_band_stats_df['Std Dev'].to_numpy()
channel_mins = planet_band_stats_df['Min'].to_numpy()
channel_maxes = planet_band_stats_df['Max'].to_numpy()
channel_mean_plus_3std = planet_band_stats_df['Mean+3Std'].to_numpy()

np.save(os.path.join(STATS_DIR, f"channel_means_{stats_desc}.npy"), channel_means)
np.save(os.path.join(STATS_DIR, f"channel_stds_{stats_desc}.npy"), channel_stds)
np.save(os.path.join(STATS_DIR, f"channel_mins_{stats_desc}.npy"), channel_mins)
np.save(os.path.join(STATS_DIR, f"channel_maxes_{stats_desc}.npy"), channel_maxes)
np.save(os.path.join(STATS_DIR, f"channel_mean_plus_3std_{stats_desc}.npy"), channel_mean_plus_3std)

# Single combined file with everything, including band names, for easy reference
planet_stats_combined = {
    'band_names': PLANET_BAND_NAMES,
    'means': channel_means,
    'stds': channel_stds,
    'mins': channel_mins,
    'maxes': channel_maxes,
    'mean_plus_3std': channel_mean_plus_3std,
    'n_sampled': planet_band_stats_df['N Sampled'].to_numpy(),
    'source_files': planet_8b_files,
}
combined_path = os.path.join(STATS_DIR, f"planet_stats_combined_{stats_desc}.npy")
np.save(combined_path, planet_stats_combined, allow_pickle=True)

print(f"Saved per-channel stats arrays to {STATS_DIR} (desc='{stats_desc}')")
print(f"Combined dict saved to: {combined_path}")
print("  -> reload with: stats = np.load(path, allow_pickle=True).item()")

In [ ]:
# ==========================================
# 8. COMPUTE SCALED-SPACE (0-1) NRG STATS
# Matches the Maxar training preprocessing: minmax scale each band to [0,1]
# using (0, mean+3std) as the bound, clip to [0,1], THEN compute mean/std of
# the scaled values (not the raw DN values) for the z-score step.
# ==========================================

import glob
import os
import numpy as np
import rasterio

STATS_DIR = "./numpy_stats/"
os.makedirs(STATS_DIR, exist_ok=True)

# Rasterio (1-indexed) band numbers for the 8-band Planet SuperDove product
NRG_BAND_NUMBERS = {'nir': 8, 'red': 6, 'green': 4}
NRG_ORDER = ['nir', 'red', 'green']

# Reuse the mean+3std bounds already computed and saved in cell 7
planet_stats = np.load(
    f"./numpy_stats/planet_stats_combined_{stats_desc}.npy",  # must match cell 7's stats_desc
    allow_pickle=True
).item()
NIR_IDX, RED_IDX, GREEN_IDX = 7, 5, 3
nrg_bounds = {
    'nir': float(planet_stats['mean_plus_3std'][NIR_IDX]),
    'red': float(planet_stats['mean_plus_3std'][RED_IDX]),
    'green': float(planet_stats['mean_plus_3std'][GREEN_IDX]),
}
print(f"Scaling bounds (0, mean+3std): {nrg_bounds}")

def sample_scaled_nrg_stats(tif_paths, bounds, sample_size=1_000_000, nodata=0, seed=0):
    """
    Sample pixels for NIR/Red/Green, apply the same minmax(0,bound)->clip[0,1]
    scaling used at inference time, and compute mean/std of the SCALED values.
    """
    rng = np.random.default_rng(seed)
    per_file_budget = max(1, sample_size // len(tif_paths))
    scaled_samples = {k: [] for k in NRG_ORDER}

    for tif_path in tif_paths:
        with rasterio.open(tif_path) as src:
            for name in NRG_ORDER:
                band = src.read(NRG_BAND_NUMBERS[name]).astype(np.float32).ravel()
                valid = band[band != nodata]
                if valid.size == 0:
                    continue
                if valid.size > per_file_budget:
                    idx = rng.choice(valid.size, per_file_budget, replace=False)
                    valid = valid[idx]
                bound = bounds[name]
                scaled = np.clip((valid - 0) / (bound - 0), 0, 1)
                scaled_samples[name].append(scaled)
        print(f"  sampled {os.path.basename(tif_path)}")

    return {k: (np.concatenate(v) if v else np.array([])) for k, v in scaled_samples.items()}


planet_8b_files = sorted(glob.glob(os.path.join(INPUT_DIR, '*_8b_sr.tif')))
scaled_nrg_samples = sample_scaled_nrg_stats(planet_8b_files, nrg_bounds, sample_size=1_000_000)

scaled_means = np.array([scaled_nrg_samples[k].mean() for k in NRG_ORDER], dtype=np.float32)
scaled_stds = np.array([scaled_nrg_samples[k].std() for k in NRG_ORDER], dtype=np.float32)

print("\nScaled [0,1] NRG stats (order NIR, Red, Green):")
for name, m, s in zip(NRG_ORDER, scaled_means, scaled_stds):
    print(f"  {name}: mean={m:.4f}, std={s:.4f}")

scaled_stats_desc = "planet_nrg_scaled01"
np.save(os.path.join(STATS_DIR, f"channel_means_{scaled_stats_desc}.npy"), scaled_means)
np.save(os.path.join(STATS_DIR, f"channel_stds_{scaled_stats_desc}.npy"), scaled_stds)
print(f"\nSaved to {STATS_DIR} (desc='{scaled_stats_desc}')")